IMPORT LIBRARY

In [4]:
import os
import json
import pickle
import random

import numpy as np
import tensorflow as tf

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

SET SEED

In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

LOAD DATASET

In [6]:
dataset_path = "../data/clean_recipes_5000.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    recipes = json.load(f)

print("Jumlah dataset:", len(recipes))

Jumlah dataset: 5000


ANALISIS QUALITY SCORE

In [7]:
scores = []

for item in recipes:
    score = item.get("Quality Score", None)

    if score is not None:
        scores.append(float(score))

scores = np.array(scores)

print(f"Jumlah data Quality Score: {len(scores)}")
print(f"Min: {scores.min():.4f}")
print(f"Max: {scores.max():.4f}")
print(f"Mean: {scores.mean():.4f}")
print(f"Median: {np.median(scores):.4f}")
print(f"Std: {scores.std():.4f}")

print("\nContoh 20 score pertama:")
print(scores[:20])

baseline_pred = np.full_like(scores, scores.mean())
baseline_mae = np.mean(np.abs(scores - baseline_pred))

print(f"\nBaseline MAE: {baseline_mae:.4f}")

Jumlah data Quality Score: 5000
Min: 0.0842
Max: 0.7997
Mean: 0.3000
Median: 0.2934
Std: 0.0687

Contoh 20 score pertama:
[0.7997 0.4689 0.4827 0.4223 0.355  0.4114 0.2336 0.4004 0.4036 0.5982
 0.4532 0.7397 0.3372 0.3399 0.3861 0.3748 0.4213 0.3162 0.2967 0.2667]

Baseline MAE: 0.0520


FEATURE ENGINEERING

In [8]:
text_features = []
numeric_features = []
quality_scores = []

for recipe in recipes:
    title = recipe.get("Title Cleaned", "")
    ingredients = recipe.get("Ingredients Cleaned", "")
    raw_ingredients = recipe.get("Ingredients", "")
    steps = recipe.get("Steps", "")
    category = recipe.get("Category", "")
    quality_score = recipe.get("Quality Score", None)

    if (
        ingredients
        and isinstance(ingredients, str)
        and ingredients.strip()
        and quality_score is not None
    ):
        combined_text = f"{title} {ingredients} {category}"

        loves = float(recipe.get("Loves", 0))
        total_ingredients = float(recipe.get("Total Ingredients", 0))
        total_steps = float(recipe.get("Total Steps", 0))

        title_length = len(title)
        ingredients_length = len(ingredients)
        raw_ingredients_length = len(raw_ingredients)
        steps_length = len(steps)

        ingredients_word_count = len(ingredients.split())
        steps_word_count = len(steps.split())

        numeric_features.append([
            np.log1p(loves),
            total_ingredients,
            total_steps,
            title_length,
            ingredients_length,
            raw_ingredients_length,
            steps_length,
            ingredients_word_count,
            steps_word_count
        ])

        text_features.append(combined_text.lower())
        quality_scores.append(float(quality_score))

print(f"Jumlah data valid: {len(text_features)}")

vectorizer = TfidfVectorizer(
    max_features=7000,
    min_df=1,
    ngram_range=(1, 2)
)

X_text = vectorizer.fit_transform(text_features)
X_text = X_text.toarray().astype(np.float32)

scaler = StandardScaler()
X_numeric = scaler.fit_transform(numeric_features).astype(np.float32)

X = np.concatenate(
    [X_text, X_numeric],
    axis=1
)

y = np.array(
    quality_scores,
    dtype=np.float32
)

print(f"Bentuk TF-IDF Matrix: {X_text.shape}")
print(f"Bentuk fitur numerik: {X_numeric.shape}")
print(f"Bentuk fitur gabungan: {X.shape}")
print(f"Bentuk target Quality Score: {y.shape}")

Jumlah data valid: 5000
Bentuk TF-IDF Matrix: (5000, 7000)
Bentuk fitur numerik: (5000, 9)
Bentuk fitur gabungan: (5000, 7009)
Bentuk target Quality Score: (5000,)


SPLIT DATASET

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED
)

print(f"Jumlah data training: {len(X_train)}")
print(f"Jumlah data validation: {len(X_val)}")
print(f"Jumlah data testing: {len(X_test)}")

Jumlah data training: 3500
Jumlah data validation: 750
Jumlah data testing: 750


BUILD REGRESSION MODEL

In [10]:
input_layer = tf.keras.layers.Input(shape=(X.shape[1],))

x = tf.keras.layers.Dense(
    256,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(0.00001)
)(input_layer)

x = tf.keras.layers.Dropout(0.1)(x)

x = tf.keras.layers.Dense(
    128,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(0.00001)
)(x)

x = tf.keras.layers.Dropout(0.1)(x)

x = tf.keras.layers.Dense(
    64,
    activation="relu"
)(x)

output_layer = tf.keras.layers.Dense(
    1,
    activation="linear"
)(x)

model = tf.keras.Model(
    inputs=input_layer,
    outputs=output_layer
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 7009)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,794,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,835,777 (7.00 MB)

 Trainable params: 1,835,777 (7.00 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0005
)

loss_fn = tf.keras.losses.MeanAbsoluteError()

batch_size = 32
epochs = 30

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(1000, seed=SEED)
    .batch(batch_size)
)

print(f"Batch size: {batch_size}")
print(f"Epochs: {epochs}")
print(f"Jumlah batch training: {len(train_dataset)}")

Batch size: 32
Epochs: 30
Jumlah batch training: 110


TENSORBOARD SETUP

In [12]:
log_dir = "../logs/quality_score"

writer = tf.summary.create_file_writer(
    log_dir
)

print(f"TensorBoard logs Quality Score: {log_dir}")

TensorBoard logs Quality Score: ../logs/quality_score


CUSTOM TRAINING LOOP & GRADIENTTAPE

In [13]:
best_val_mae = float("inf")
best_weights = None
patience = 8
wait = 0

for epoch in range(epochs):
    epoch_losses = []
    epoch_maes = []

    for batch_x, batch_y in train_dataset:
        batch_y = tf.reshape(batch_y, (-1, 1))

        with tf.GradientTape() as tape:
            predictions = model(
                batch_x,
                training=True
            )

            loss = loss_fn(
                batch_y,
                predictions
            )

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(
                gradients,
                model.trainable_variables
            )
        )

        batch_mae = mean_absolute_error(
            batch_y.numpy().reshape(-1),
            predictions.numpy().reshape(-1)
        )

        epoch_losses.append(loss.numpy())
        epoch_maes.append(batch_mae)

    train_loss = np.mean(epoch_losses)
    train_mae = np.mean(epoch_maes)

    val_predictions = model(
        X_val,
        training=False
    ).numpy().reshape(-1)

    val_mae = mean_absolute_error(
        y_val,
        val_predictions
    )

    val_loss = val_mae

    print(f"Epoch {epoch + 1}/{epochs}")
    print(
        f"{len(train_dataset)}/{len(train_dataset)} "
        f"- loss: {train_loss:.4f} "
        f"- mae: {train_mae:.4f} "
        f"- val_loss: {val_loss:.4f} "
        f"- val_mae: {val_mae:.4f}"
    )

    with writer.as_default():
        tf.summary.scalar("loss", train_loss, step=epoch)
        tf.summary.scalar("mae", train_mae, step=epoch)
        tf.summary.scalar("val_loss", val_loss, step=epoch)
        tf.summary.scalar("val_mae", val_mae, step=epoch)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_weights = model.get_weights()
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping aktif")
        break

if best_weights is not None:
    model.set_weights(best_weights)

Epoch 1/30
110/110 - loss: 0.0481 - mae: 0.0481 - val_loss: 0.0179 - val_mae: 0.0179
Epoch 2/30
110/110 - loss: 0.0217 - mae: 0.0217 - val_loss: 0.0201 - val_mae: 0.0201
Epoch 3/30
110/110 - loss: 0.0158 - mae: 0.0158 - val_loss: 0.0140 - val_mae: 0.0140
Epoch 4/30
110/110 - loss: 0.0125 - mae: 0.0125 - val_loss: 0.0095 - val_mae: 0.0095
Epoch 5/30
110/110 - loss: 0.0105 - mae: 0.0105 - val_loss: 0.0098 - val_mae: 0.0098
Epoch 6/30
110/110 - loss: 0.0096 - mae: 0.0096 - val_loss: 0.0077 - val_mae: 0.0077
Epoch 7/30
110/110 - loss: 0.0091 - mae: 0.0091 - val_loss: 0.0116 - val_mae: 0.0116
Epoch 8/30
110/110 - loss: 0.0082 - mae: 0.0082 - val_loss: 0.0084 - val_mae: 0.0084
Epoch 9/30
110/110 - loss: 0.0079 - mae: 0.0079 - val_loss: 0.0085 - val_mae: 0.0085
Epoch 10/30
110/110 - loss: 0.0078 - mae: 0.0078 - val_loss: 0.0087 - val_mae: 0.0087
Epoch 11/30
110/110 - loss: 0.0074 - mae: 0.0074 - val_loss: 0.0088 - val_mae: 0.0088
Epoch 12/30
110/110 - loss: 0.0073 - mae: 0.0073 - val_loss: 0.

EVALUASI DATA TESTING

In [14]:
test_predictions = model(
    X_test,
    training=False
).numpy().reshape(-1)

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_loss = test_mae

print("\nHASIL EVALUASI QUALITY SCORE")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

if test_mae <= 0.02:
    print("Status MAE: MEMENUHI syarat MAE maksimal 0.02")
else:
    print("Status MAE: BELUM MEMENUHI syarat MAE maksimal 0.02")


HASIL EVALUASI QUALITY SCORE
Test Loss: 0.0081
Test MAE: 0.0081
Status MAE: MEMENUHI syarat MAE maksimal 0.02


SAVE MODEL

In [15]:
models_dir = "../models"

os.makedirs(
    models_dir,
    exist_ok=True
)

quality_model_path = os.path.join(
    models_dir,
    "quality_score_model.keras"
)

model.save(
    quality_model_path
)

quality_vectorizer_path = os.path.join(
    models_dir,
    "quality_score_vectorizer.pkl"
)

with open(quality_vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)

quality_scaler_path = os.path.join(
    models_dir,
    "quality_score_scaler.pkl"
)

with open(quality_scaler_path, "wb") as f:
    pickle.dump(scaler, f)

writer.flush()
writer.close()

print(f"Model Quality Score disimpan: {quality_model_path}")
print(f"Vectorizer Quality Score disimpan: {quality_vectorizer_path}")
print(f"Scaler Quality Score disimpan: {quality_scaler_path}")
print(f"TensorBoard logs Quality Score: {log_dir}")

Model Quality Score disimpan: ../models\quality_score_model.keras
Vectorizer Quality Score disimpan: ../models\quality_score_vectorizer.pkl
Scaler Quality Score disimpan: ../models\quality_score_scaler.pkl
TensorBoard logs Quality Score: ../logs/quality_score


In [16]:
"""
Jalankan TensorBoard dengan perintah berikut di terminal:

cd "quest\\Path AI"
tensorboard --logdir logs --port 6006

Lalu buka:
http://localhost:6006

Catatan:
- Grafik classification berasal dari logs/classification
- Grafik quality_score berasal dari logs/quality_score
"""

'\nJalankan TensorBoard dengan perintah berikut di terminal:\n\ncd "quest\\Path AI"\ntensorboard --logdir logs --port 6006\n\nLalu buka:\nhttp://localhost:6006\n\nCatatan:\n- Grafik classification berasal dari logs/classification\n- Grafik quality_score berasal dari logs/quality_score\n'